# GridLock Hackathon 2.0 — Traffic Demand Prediction
## Complete ML Pipeline

**Target:** `max(0, 100 x R2)` on demand predictions  
**Dataset:** 77,299 train rows · 41,778 test rows · 10 features  
**Models:** LightGBM · XGBoost · CatBoost · Ensemble · Optuna tuning

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score, mean_squared_error
import lightgbm as lgb
from lightgbm import LGBMRegressor, early_stopping as lgb_early_stop, log_evaluation as lgb_log
import xgboost as xgb
from catboost import CatBoostRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import pygeohash as pgh

plt.style.use('ggplot')
pd.set_option('display.max_columns', 50)
np.random.seed(42)
print('All imports successful!')

## Load Data

In [ ]:
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')
print(f'Train: {train.shape}  |  Test: {test.shape}')
print('\nTrain dtypes:\n', train.dtypes)
train.head(3)

---
## Step 1: Exploratory Data Analysis

### 1a. Demand Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(train['demand'], bins=60, edgecolor='black', alpha=0.75, color='steelblue')
axes[0].set_title('Demand Distribution', fontsize=13)
axes[0].set_xlabel('Demand'); axes[0].set_ylabel('Frequency')

axes[1].hist(np.log1p(train['demand']), bins=60, edgecolor='black', alpha=0.75, color='darkorange')
axes[1].set_title('log1p(Demand) Distribution', fontsize=13)
axes[1].set_xlabel('log1p(Demand)'); axes[1].set_ylabel('Frequency')

plt.suptitle('Traffic Demand Distribution', fontsize=15)
plt.tight_layout()
plt.savefig('demand_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

print('\nDemand Statistics:')
print(train['demand'].describe())
print(f'\nRows with demand > 0.5: {(train["demand"] > 0.5).sum()} ({100*(train["demand"] > 0.5).mean():.1f}%)')

### 1b. Unique Values

In [ ]:
for col in ['day', 'RoadType', 'Weather', 'LargeVehicles', 'Landmarks']:
    vals = train[col].unique()
    print(f'{col:15s} ({len(vals)} unique): {vals}')
ts_unique = sorted(train['timestamp'].unique())
print(f'\ntimestamp: {len(ts_unique)} unique values | sample: {ts_unique[:8]}')

### 1c. Missing Values

In [ ]:
print('Missing values - TRAIN:')
miss_tr = train.isnull().sum()
print(miss_tr[miss_tr > 0])
print(f'\nTotal train missing: {train.isnull().sum().sum()}')

print('\nMissing values - TEST:')
miss_te = test.isnull().sum()
print(miss_te[miss_te > 0])
print(f'\nTotal test missing: {test.isnull().sum().sum()}')

### 1d. Correlation with Demand

In [ ]:
print('Pearson correlation with demand:')
for col in ['NumberofLanes', 'Temperature']:
    corr = train[col].corr(train['demand'])
    print(f'  {col:<20s}: {corr:+.4f}')

tmp = train.copy()
tmp['hour'] = tmp['timestamp'].apply(lambda x: int(x.split(':')[0]))
hourly = tmp.groupby('hour')['demand'].mean()

fig, ax = plt.subplots(figsize=(12, 4))
hourly.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Mean Demand by Hour of Day')
ax.set_xlabel('Hour'); ax.set_ylabel('Mean Demand')
plt.tight_layout()
plt.savefig('demand_by_hour.png', dpi=100, bbox_inches='tight')
plt.show()

### 1e. Mean Demand by Category

In [ ]:
for col in ['RoadType', 'Weather', 'LargeVehicles', 'Landmarks', 'NumberofLanes']:
    print(f'\nMean demand by {col}:')
    print(train.groupby(col, dropna=False)['demand'].agg(['mean','std','count']).round(4))

---
## Step 2: Feature Engineering

A `FeatureEngineer` class fits on train data only. All statistics (geohash means, imputation fills, encodings) are learned on train and applied to test to prevent leakage.

In [ ]:
class FeatureEngineer:
    'Fits on train, applies identically to test (zero leakage).'

    # ── helpers ──────────────────────────────────────────────────────

    @staticmethod
    def _decode_gh(gh):
        try:
            lat, lng, lat_err, lng_err = pgh.decode_exactly(gh)
            return float(lat), float(lng), float(lat_err), float(lng_err)
        except Exception:
            return 0.0, 0.0, 0.0, 0.0

    @staticmethod
    def _parse_ts(ts):
        h, m = ts.split(':')
        return int(h), int(m)

    @staticmethod
    def _time_of_day(h):
        if   h <= 5:  return 0   # night 0-5
        elif h <= 11: return 1   # morning 6-11
        elif h <= 17: return 2   # afternoon 12-17
        else:         return 3   # evening 18-23

    # ── public API ───────────────────────────────────────────────────

    def fit_transform(self, df):
        'Fit statistics on df (train) and return engineered features.'
        df = df.copy()

        # 2g: compute imputation stats on train
        self._temp_med_global = float(df['Temperature'].median())
        self._temp_med_gh     = df.groupby('geohash')['Temperature'].median()

        self._rt_mode_global  = (df['RoadType'].mode().iloc[0]
                                 if len(df['RoadType'].mode()) else 'Residential')
        self._rt_mode_gh      = df.groupby('geohash')['RoadType'].apply(
            lambda x: x.mode().iloc[0] if len(x.mode()) else self._rt_mode_global)

        self._wx_mode_global  = (df['Weather'].mode().iloc[0]
                                 if len(df['Weather'].mode()) else 'Sunny')
        self._wx_mode_gh      = df.groupby('geohash')['Weather'].apply(
            lambda x: x.mode().iloc[0] if len(x.mode()) else self._wx_mode_global)

        df = self._impute(df)

        # encoding parameters from train
        self._max_day  = int(df['day'].max())
        self._rt_cols  = sorted(df['RoadType'].dropna().unique().tolist())
        self._wx_cols  = sorted(df['Weather'].dropna().unique().tolist())

        # geohash target-encoding stats (demand column exists only on train)
        self._gh_mean   = df.groupby('geohash')['demand'].mean()
        self._gh_median = df.groupby('geohash')['demand'].median()
        self._gh_std    = df.groupby('geohash')['demand'].std().fillna(0.0)
        self._g_mean    = float(df['demand'].mean())
        self._g_median  = float(df['demand'].median())
        self._g_std     = float(df['demand'].std())

        self.fitted = True
        return self._transform(df)

    def transform(self, df):
        'Apply fitted transformations to new data (no leakage).'
        assert self.fitted, 'Call fit_transform first'
        df = df.copy()
        df = self._impute(df)
        return self._transform(df)

    # ── internals ────────────────────────────────────────────────────

    def _impute(self, df):
        def fill_col(row, col, gh_map, global_val):
            if pd.isna(row[col]):
                return gh_map.get(row['geohash'], global_val)
            return row[col]

        df['Temperature'] = df.apply(
            fill_col, axis=1,
            args=('Temperature', self._temp_med_gh, self._temp_med_global)
        ).fillna(self._temp_med_global)

        df['RoadType'] = df.apply(
            fill_col, axis=1,
            args=('RoadType', self._rt_mode_gh, self._rt_mode_global)
        ).fillna(self._rt_mode_global)

        df['Weather'] = df.apply(
            fill_col, axis=1,
            args=('Weather', self._wx_mode_gh, self._wx_mode_global)
        ).fillna(self._wx_mode_global)

        return df

    def _transform(self, df):
        # 2a: geohash decoding
        geo = df['geohash'].apply(self._decode_gh)
        df['geohash_lat']     = geo.apply(lambda x: x[0])
        df['geohash_lng']     = geo.apply(lambda x: x[1])
        df['geohash_lat_err'] = geo.apply(lambda x: x[2])
        df['geohash_lng_err'] = geo.apply(lambda x: x[3])

        # 2b: timestamp features
        ts = df['timestamp'].apply(self._parse_ts)
        df['hour']         = ts.apply(lambda x: x[0])
        df['minute']       = ts.apply(lambda x: x[1])
        df['hour_sin']     = np.sin(2 * np.pi * df['hour']   / 24)
        df['hour_cos']     = np.cos(2 * np.pi * df['hour']   / 24)
        df['minute_sin']   = np.sin(2 * np.pi * df['minute'] / 60)
        df['minute_cos']   = np.cos(2 * np.pi * df['minute'] / 60)
        df['is_rush_hour'] = df['hour'].isin([7,8,9,17,18,19]).astype(int)
        df['time_of_day']  = df['hour'].apply(self._time_of_day)

        # 2c: day features
        df['day_sin']   = np.sin(2 * np.pi * df['day'] / self._max_day)
        df['day_cos']   = np.cos(2 * np.pi * df['day'] / self._max_day)
        df['day_mod_7'] = df['day'] % 7

        # 2d: categorical encoding
        df['LargeVehicles'] = (df['LargeVehicles'] == 'Allowed').astype(int)
        df['Landmarks']     = (df['Landmarks']     == 'Yes').astype(int)
        for rt in self._rt_cols:
            df[f'RoadType_{rt}'] = (df['RoadType'] == rt).astype(int)
        for wx in self._wx_cols:
            df[f'Weather_{wx}']  = (df['Weather']  == wx).astype(int)

        # 2e: geohash target encoding (uses train stats only)
        df['geohash_mean_demand']   = df['geohash'].map(self._gh_mean  ).fillna(self._g_mean)
        df['geohash_median_demand'] = df['geohash'].map(self._gh_median).fillna(self._g_median)
        df['geohash_std_demand']    = df['geohash'].map(self._gh_std   ).fillna(self._g_std)

        # 2f: interaction features
        rt_num = {'Highway': 3, 'Street': 2, 'Residential': 1}
        df['RoadType_numeric'] = df['RoadType'].map(rt_num).fillna(0).astype(float)
        df['lanes_x_roadtype'] = df['NumberofLanes'] * df['RoadType_numeric']

        wx_num = {'Sunny': 1, 'Cloudy': 2, 'Rainy': 3, 'Foggy': 4, 'Snowy': 5}
        df['Weather_numeric']  = df['Weather'].map(wx_num).fillna(1).astype(float)
        df['temp_x_weather']   = df['Temperature'] * df['Weather_numeric']

        df['location_time'] = df['geohash_mean_demand'] * df['hour_sin']

        return df

print('FeatureEngineer class defined.')

### Apply Feature Engineering to Train

In [ ]:
fe       = FeatureEngineer()
train_fe = fe.fit_transform(train)
print(f'Engineered train shape: {train_fe.shape}')
print(f'\nSample engineered columns:')
train_fe[['geohash_lat','geohash_lng','hour','hour_sin',
          'geohash_mean_demand','lanes_x_roadtype','temp_x_weather',
          'location_time','demand']].head(3)

### Define Feature Columns

In [ ]:
EXCLUDE      = {'Index', 'geohash', 'timestamp', 'demand', 'RoadType', 'Weather'}
feature_cols = [c for c in train_fe.columns if c not in EXCLUDE]
print(f'Total features: {len(feature_cols)}')
print('\nFeature list:')
for i, c in enumerate(feature_cols, 1):
    print(f'  {i:2d}. {c}')

In [ ]:
X_all = train_fe[feature_cols].values.astype(np.float32)
y_all = train_fe['demand'].values.astype(np.float32)

assert not np.isnan(X_all).any(), 'NaN values found in X_all!'
print(f'X_all: {X_all.shape}  |  y_all: {y_all.shape}')
print(f'y range: [{y_all.min():.4f}, {y_all.max():.4f}]  mean: {y_all.mean():.4f}')

---
## Step 3: Modelling

**Time-aware split:** sort by day + time, hold out the **last 15%** as validation (no random shuffle — respects temporal ordering).

### 3a. Time-Aware Train / Validation Split

In [ ]:
sort_keys  = train_fe['day'] * 1440 + train_fe['hour'] * 60 + train_fe['minute']
sorted_idx = sort_keys.argsort().values

split_idx = int(len(sorted_idx) * 0.85)
tr_idx    = sorted_idx[:split_idx]
va_idx    = sorted_idx[split_idx:]

X_train, y_train = X_all[tr_idx], y_all[tr_idx]
X_val,   y_val   = X_all[va_idx], y_all[va_idx]

print(f'Train split : {X_train.shape}')
print(f'Val   split : {X_val.shape}')
print(f'Val demand  : mean={y_val.mean():.4f}  std={y_val.std():.4f}')

### 3b-1. LightGBM

In [ ]:
lgbm_model = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=127,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    n_jobs=-1,
    random_state=42,
    verbose=-1
)

lgbm_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb_early_stop(50, verbose=False), lgb_log(100)]
)

lgbm_pred_val = lgbm_model.predict(X_val)
lgbm_r2       = r2_score(y_val, lgbm_pred_val)
lgbm_rmse     = np.sqrt(mean_squared_error(y_val, lgbm_pred_val))

print(f'LightGBM  |  Val R2 = {lgbm_r2:.6f}  |  RMSE = {lgbm_rmse:.6f}')
print(f'Hackathon score estimate: {max(0, lgbm_r2 * 100):.2f}')
print(f'Best iteration: {lgbm_model.best_iteration_}')

lgbm_imp = pd.DataFrame({'feature': feature_cols,
                          'importance': lgbm_model.feature_importances_}
                        ).sort_values('importance', ascending=False)
print('\nTop 20 features (LightGBM):')
print(lgbm_imp.head(20).to_string(index=False))

### 3b-2. XGBoost

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    early_stopping_rounds=50,
    eval_metric='rmse',
    n_jobs=-1,
    random_state=42,
    verbosity=0
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100
)

xgb_pred_val = xgb_model.predict(X_val)
xgb_r2       = r2_score(y_val, xgb_pred_val)
xgb_rmse     = np.sqrt(mean_squared_error(y_val, xgb_pred_val))

print(f'XGBoost   |  Val R2 = {xgb_r2:.6f}  |  RMSE = {xgb_rmse:.6f}')
print(f'Hackathon score estimate: {max(0, xgb_r2 * 100):.2f}')
print(f'Best iteration: {xgb_model.best_iteration}')

xgb_imp = pd.DataFrame({'feature': feature_cols,
                         'importance': xgb_model.feature_importances_}
                       ).sort_values('importance', ascending=False)
print('\nTop 20 features (XGBoost):')
print(xgb_imp.head(20).to_string(index=False))

### 3b-3. CatBoost

In [ ]:
cat_model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    min_data_in_leaf=20,
    subsample=0.8,
    colsample_bylevel=0.8,
    l2_leaf_reg=3.0,
    early_stopping_rounds=50,
    random_state=42,
    verbose=100
)

cat_model.fit(X_train, y_train, eval_set=(X_val, y_val))

cat_pred_val = cat_model.predict(X_val)
cat_r2       = r2_score(y_val, cat_pred_val)
cat_rmse     = np.sqrt(mean_squared_error(y_val, cat_pred_val))

print(f'CatBoost  |  Val R2 = {cat_r2:.6f}  |  RMSE = {cat_rmse:.6f}')
print(f'Hackathon score estimate: {max(0, cat_r2 * 100):.2f}')
print(f'Best iteration: {cat_model.best_iteration_}')

cat_imp = pd.DataFrame({'feature': feature_cols,
                         'importance': cat_model.get_feature_importance()}
                       ).sort_values('importance', ascending=False)
print('\nTop 20 features (CatBoost):')
print(cat_imp.head(20).to_string(index=False))

### 3c. Ensemble (LGBM×0.4 + XGB×0.3 + CAT×0.3)

In [ ]:
ensemble_pred_val = (0.4 * lgbm_pred_val
                   + 0.3 * xgb_pred_val
                   + 0.3 * cat_pred_val)
ensemble_r2   = r2_score(y_val, ensemble_pred_val)
ensemble_rmse = np.sqrt(mean_squared_error(y_val, ensemble_pred_val))

print('=' * 55)
print('Ensemble (LGBM x0.4 + XGB x0.3 + CAT x0.3)')
print(f'  Val R2   = {ensemble_r2:.6f}')
print(f'  Val RMSE = {ensemble_rmse:.6f}')
print(f'  Score    = {max(0, ensemble_r2 * 100):.2f}')
print('=' * 55)
print(f'\nSingle models:')
print(f'  LightGBM : {lgbm_r2:.6f}')
print(f'  XGBoost  : {xgb_r2:.6f}')
print(f'  CatBoost : {cat_r2:.6f}')

single_scores    = {'LightGBM': lgbm_r2, 'XGBoost': xgb_r2, 'CatBoost': cat_r2}
best_single_name = max(single_scores, key=single_scores.get)
best_single_r2   = single_scores[best_single_name]
use_ensemble     = ensemble_r2 > best_single_r2

print(f'\nBest single model : {best_single_name} (R2={best_single_r2:.6f})')
print(f'Use ensemble for submission: {use_ensemble}')

---
## Step 4: Hyperparameter Tuning with Optuna

Tune the best single model for 50 trials, maximising R2 on the validation set.

In [ ]:
print(f'Tuning: {best_single_name}')

def objective_lgbm(trial):
    params = dict(
        n_estimators      = 1000,
        learning_rate     = trial.suggest_float('learning_rate',    0.01,  0.15,  log=True),
        num_leaves        = trial.suggest_int(  'num_leaves',       31,    255),
        min_child_samples = trial.suggest_int(  'min_child_samples',5,     100),
        subsample         = trial.suggest_float('subsample',        0.5,   1.0),
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.5,   1.0),
        reg_alpha         = trial.suggest_float('reg_alpha',        1e-3,  10.0, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda',       1e-3,  10.0, log=True),
        n_jobs=-1, random_state=42, verbose=-1
    )
    m = LGBMRegressor(**params)
    m.fit(X_train, y_train, eval_set=[(X_val, y_val)],
          callbacks=[lgb_early_stop(50, verbose=False)])
    return r2_score(y_val, m.predict(X_val))

def objective_xgb(trial):
    params = dict(
        n_estimators        = 1000,
        learning_rate       = trial.suggest_float('learning_rate',   0.01, 0.15, log=True),
        max_depth           = trial.suggest_int(  'max_depth',       3,    10),
        min_child_weight    = trial.suggest_int(  'min_child_weight',1,    20),
        subsample           = trial.suggest_float('subsample',       0.5,  1.0),
        colsample_bytree    = trial.suggest_float('colsample_bytree',0.5,  1.0),
        reg_alpha           = trial.suggest_float('reg_alpha',       1e-3, 10.0, log=True),
        reg_lambda          = trial.suggest_float('reg_lambda',      1e-3, 10.0, log=True),
        early_stopping_rounds=50, eval_metric='rmse',
        n_jobs=-1, random_state=42, verbosity=0
    )
    m = xgb.XGBRegressor(**params)
    m.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return r2_score(y_val, m.predict(X_val))

def objective_cat(trial):
    params = dict(
        iterations        = 1000,
        learning_rate     = trial.suggest_float('learning_rate',    0.01, 0.15, log=True),
        depth             = trial.suggest_int(  'depth',            4,    10),
        min_data_in_leaf  = trial.suggest_int(  'min_data_in_leaf', 1,    50),
        subsample         = trial.suggest_float('subsample',        0.5,  1.0),
        colsample_bylevel = trial.suggest_float('colsample_bylevel',0.5,  1.0),
        l2_leaf_reg       = trial.suggest_float('l2_leaf_reg',      1.0,  10.0),
        early_stopping_rounds=50, random_state=42, verbose=0
    )
    m = CatBoostRegressor(**params)
    m.fit(X_train, y_train, eval_set=(X_val, y_val))
    return r2_score(y_val, m.predict(X_val))

obj_map = {'LightGBM': objective_lgbm, 'XGBoost': objective_xgb, 'CatBoost': objective_cat}

study = optuna.create_study(direction='maximize',
                             sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(obj_map[best_single_name], n_trials=50, show_progress_bar=True)

print(f'\nBest trial R2 : {study.best_value:.6f}')
print('Best params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

### Retrain Tuned Model on Full Training Data

In [ ]:
best_params = study.best_params.copy()
print(f'Retraining tuned {best_single_name} on full training data ({X_all.shape[0]} rows)...')

if best_single_name == 'LightGBM':
    n_iters = int(lgbm_model.best_iteration_ * 1.2) + 1
    tuned_model = LGBMRegressor(
        n_estimators=n_iters,
        **{k: v for k, v in best_params.items()},
        n_jobs=-1, random_state=42, verbose=-1
    )
    tuned_model.fit(X_all, y_all)

elif best_single_name == 'XGBoost':
    n_iters = int(xgb_model.best_iteration * 1.2) + 1
    tuned_model = xgb.XGBRegressor(
        n_estimators=n_iters,
        **{k: v for k, v in best_params.items() if k not in ('early_stopping_rounds','eval_metric')},
        n_jobs=-1, random_state=42, verbosity=0
    )
    tuned_model.fit(X_all, y_all)

else:  # CatBoost
    n_iters = int(cat_model.best_iteration_ * 1.2) + 1
    tuned_model = CatBoostRegressor(
        iterations=n_iters,
        **{k: v for k, v in best_params.items() if k != 'early_stopping_rounds'},
        random_state=42, verbose=0
    )
    tuned_model.fit(X_all, y_all)

tuned_pred_val = tuned_model.predict(X_val)
tuned_r2       = r2_score(y_val, tuned_pred_val)
print(f'\nTuned {best_single_name} (on val — for reference): R2 = {tuned_r2:.6f}')
print(f'Hackathon score estimate: {max(0, tuned_r2 * 100):.2f}')

---
## Step 5: Final Predictions

Apply the **same** feature engineering pipeline (fitted on train) to the test set, then generate clipped predictions.

In [ ]:
test_fe   = fe.transform(test)
X_test    = test_fe[feature_cols].values.astype(np.float32)
print(f'Test features shape : {X_test.shape}')

nan_count = int(np.isnan(X_test).sum())
print(f'NaN values in X_test: {nan_count}')
if nan_count > 0:
    nan_ser = pd.DataFrame(X_test, columns=feature_cols).isnull().sum()
    print(nan_ser[nan_ser > 0])
    X_test = np.nan_to_num(X_test, nan=0.0)
    print('Filled remaining NaN with 0')

In [ ]:
max_train_demand = float(train['demand'].max())
print(f'Max training demand : {max_train_demand:.6f}')

if use_ensemble:
    lgbm_test = lgbm_model.predict(X_test)
    xgb_test  = xgb_model.predict(X_test)
    cat_test  = cat_model.predict(X_test)
    final_preds = 0.4 * lgbm_test + 0.3 * xgb_test + 0.3 * cat_test
    print('Using ensemble for final predictions')
else:
    final_preds = tuned_model.predict(X_test)
    print(f'Using tuned {best_single_name} for final predictions')

final_preds = np.clip(final_preds, 0.0, max_train_demand)

print(f'\nFinal prediction stats:')
print(f'  min  : {final_preds.min():.6f}')
print(f'  max  : {final_preds.max():.6f}')
print(f'  mean : {final_preds.mean():.6f}')
print(f'  std  : {final_preds.std():.6f}')

In [ ]:
submission = pd.DataFrame({'Index': test['Index'], 'demand': final_preds})
assert submission.shape == (41778, 2), f'Shape mismatch: {submission.shape}'
submission.to_csv('submission.csv', index=False)

print(f'submission.csv saved  |  shape: {submission.shape}')
print('\nFirst 5 rows:')
print(submission.head())
print('\nPredictions describe():')
print(submission['demand'].describe())

---
## Step 6: Validation Report

In [ ]:
print('=' * 60)
print('  GRIDLOCK HACKATHON 2.0 - VALIDATION REPORT')
print('=' * 60)

models_summary = {
    'LightGBM (baseline)'    : lgbm_r2,
    'XGBoost  (baseline)'    : xgb_r2,
    'CatBoost (baseline)'    : cat_r2,
    'Ensemble (0.4/0.3/0.3)' : ensemble_r2,
    f'Tuned {best_single_name}'  : tuned_r2,
}

print('\n{:<30s} {:>10s}  {:>8s}'.format('Model', 'Val R2', 'Score'))
print('-' * 55)
for name, r2 in sorted(models_summary.items(), key=lambda x: -x[1]):
    score  = max(0, r2 * 100)
    marker = '  <- best' if r2 == max(models_summary.values()) else ''
    print(f'  {name:<28s} {r2:.6f}  {score:6.2f}{marker}')

final_r2 = ensemble_r2 if use_ensemble else tuned_r2
print(f'\nFinal model for submission : {"Ensemble" if use_ensemble else f"Tuned {best_single_name}"}')
print(f'Estimated hackathon score  : {max(0, final_r2 * 100):.2f} / 100')

print('\nTop 5 Most Important Features (LightGBM):')
for _, row in lgbm_imp.head(5).iterrows():
    print(f'  {row["feature"]:<30s} {row["importance"]:.2f}')

print(f'\nSubmission file  : submission.csv')
print(f'Submission shape : {submission.shape}')
print('=' * 60)